In [1]:
import os
import json
import csv
import pandas as pd
from tqdm import tqdm

In [2]:
json_path = "data/json"
csv_path = "data/csv"

try:
    restaurants_id_set = set(pd.read_csv('data/csv/restaurants_id.csv')["restaurants_id"])
    print(f"Loaded {len(restaurants_id_set)} restaurant IDs for filtering.")
except FileNotFoundError:
    print(f"Error: The file 'data/csv/restaurants_id.csv' was not found. Please check the path.")
except Exception as e:
    print(f"An unexpected error occurred while reading the CSV: {e}")

Loaded 51928 restaurant IDs for filtering.


In [3]:
type(restaurants_id_set)

set

In [4]:
restaurants_id_set

{'cycBI4nrs7hONELNLPnfew',
 'TqHEgalpAksfVJvx8INuvg',
 'BPCauBeb0Gxk_MP3TnJjhA',
 'yih0BSPRXzPlc8qVNlYs-g',
 'eh22smcZ8-n7RgNnVYfYMA',
 'GXhExRdP5UUDhoxTRWQfcw',
 'G4VYdDhZ0lxzQ0OvUhWfEg',
 'ELZjb4effPxfDfdXqE_sKw',
 'yuNf6vYaIZYe88g6pouAiA',
 '488xZxYwdTQD6VNUHQk8XQ',
 'i8epJgmh3rUonQDKx1g3hw',
 'VHGr0eHPYSrLk8I8lstkeQ',
 'lcTR-NewLIrTTYwyAJJzUg',
 'BiPItVGi29qC0E2a2v_u2Q',
 'OSjSbJEyIkey6tcTgL9y3Q',
 'SS6QZgF1jsKMeN5QFZvIkQ',
 'gwfVq_Fmd3wGB31F_SX4Qw',
 'CuQEKyYv9Sz35n64ibO-oQ',
 'khv9-Vqn3x514ZCRQ1121A',
 'FRr2a55hbdZBUMQTE1blHA',
 'sBTUg4DINvmrfM2cLQZAjw',
 't15pblhsTMQqp7Rh7KgHVA',
 'NPVO5zTP8mbcpciaZ0-GDg',
 'onLvuAKTdabG_tSXkuqYCQ',
 'pKpu9LDo8jndikkkAHoAsw',
 'lM5TtwwucIPIzEPeDqDaEw',
 'H5cH1BbNfP5YEkytAQaYrg',
 'j6I_2GtM3PBLHSdl-fOgrA',
 'BjnJ0nBshNbmrosBHxB2eQ',
 'dNR-b-CsrFGYhMo9zLMrCw',
 'VzpxaKY-MN2Fo_wA8aiZ9w',
 'j0nfkOogyh43lFcuUMlVsQ',
 'irCGvg8WzbxXRozYRCBhwg',
 'jwg_T0_my321ag20O1g97A',
 'liy3id4Wj1AaIduwETiVSQ',
 'y7pKM3eFDZ9eRchC6SNMsA',
 'lqyYsXBAO6cVGR0cPCeydg',
 

In [ ]:
def read_ndjson_file(filepath):
    """
    Reads a large JSON Lines (NDJSON) file, treating each line as a separate 
    JSON object and returning them as a list of dictionaries.
    """
    print(f"Attempting to read data from: {filepath}")

    all_records = []
    line_count = 0
    error_count = 0
    
    try:
        # Use 'r' for reading text, and iterate line by line
        with open(filepath, 'r', encoding='utf-8') as file:
            for line in tqdm(file, desc="Loading", unit="rec", mininterval=0.5):
                line_count += 1
                # Strip whitespace/newlines from the line
                stripped_line = line.strip()
                
                if not stripped_line:
                    continue # Skip empty lines

                try:
                    # Use json.loads() because we are loading a string (the single line)
                    record = json.loads(stripped_line)
                    all_records.append(record)
                except json.JSONDecodeError as e:
                    print(f"\nSkipping malformed record on Line {line_count}:")
                    print(f"   Error details: {e}")
                    error_count += 1
        if all_records:
            print("\nSuccessfully processed JSON Lines data!")
        else:
            print("\nNo records were successfully loaded.")
            
    except FileNotFoundError:
        print(f"\nError: The file '{filepath}' was not found.")
    except Exception as e:
        print(f"\nAn unexpected error occurred during file reading: {e}")

    print(f"\n--- Summary ---")
    print(f"Total lines processed: {line_count}")
    print(f"Successfully loaded records: {len(all_records)}")
    print(f"Records skipped due to errors: {error_count}")
    
    return all_records

def convert_json_to_csv_chunked(json_filepath, csv_filepath, filter_column=None, chunk_size=100000):
    """
    Reads a large NDJSON file in chunks, optionally filters it, and appends to a CSV.
    """
    if not os.path.exists(json_filepath):
        print(f"Error: {json_filepath} not found.")
        return

    print(f"Starting conversion: {json_filepath} -> {csv_filepath}")
    os.makedirs(os.path.dirname(csv_filepath) or '.', exist_ok=True)
    
    # Remove the existing CSV if it exists so we don't infinitely append to old data
    if os.path.exists(csv_filepath):
        os.remove(csv_filepath)

    # Create a JSON reader iterator
    json_reader = pd.read_json(json_filepath, lines=True, chunksize=chunk_size)
    
    first_chunk = True
    total_processed = 0
    total_saved = 0

    try:
        for chunk in tqdm(json_reader, desc=f"Processing {os.path.basename(json_filepath)}"):
            total_processed += len(chunk)
            
            # Apply filtering if a column is specified (e.g., 'business_id')
            if filter_column and restaurants_id_set:
                # Keep only rows where the specified column matches our target IDs
                chunk = chunk[chunk[filter_column].isin(restaurants_id_set)]
            
            # If the chunk isn't empty after filtering, append it to the CSV
            if not chunk.empty:
                total_saved += len(chunk)
                # mode='a' appends to the file. header=first_chunk ensures we only write column names once.
                chunk.to_csv(csv_filepath, mode='a', index=False, header=first_chunk)
                first_chunk = False
                
        print(f"Conversion complete! Processed {total_processed} rows, saved {total_saved} rows to {csv_filepath}\n")
        
    except ValueError as e:
         print(f"Error processing chunks (likely mismatched schema): {e}")

In [7]:
# Convert Business Data (Filter by business_id)
convert_json_to_csv_chunked(
    'data/json/yelp_academic_dataset_business.json', 
    'data/csv/restaurants_business.csv', 
    filter_column='business_id'
)

Starting conversion: data/json/yelp_academic_dataset_business.json -> data/csv/restaurants_business.csv


Processing yelp_academic_dataset_business.json: 2it [00:03,  1.93s/it]

Conversion complete! Processed 150346 rows, saved 51928 rows to data/csv/restaurants_business.csv



In [8]:
# Convert Checkin Data (Filter by business_id)
convert_json_to_csv_chunked(
    'data/json/yelp_academic_dataset_checkin.json', 
    'data/csv/restaurants_checkin.csv', 
    filter_column='business_id'
)

Starting conversion: data/json/yelp_academic_dataset_checkin.json -> data/csv/restaurants_checkin.csv


Processing yelp_academic_dataset_checkin.json: 2it [00:02,  1.47s/it]

Conversion complete! Processed 131930 rows, saved 50966 rows to data/csv/restaurants_checkin.csv



In [9]:
# Convert Review Data (Filter by business_id) - This is the 5GB file!
convert_json_to_csv_chunked(
    'data/json/yelp_academic_dataset_review.json', 
    'data/csv/restaurants_review.csv', 
    filter_column='business_id'
)

Starting conversion: data/json/yelp_academic_dataset_review.json -> data/csv/restaurants_review.csv


Processing yelp_academic_dataset_review.json: 70it [01:26,  1.24s/it]

Conversion complete! Processed 6990280 rows, saved 4707976 rows to data/csv/restaurants_review.csv



In [10]:
# Convert Tip Data (Filter by business_id)
convert_json_to_csv_chunked(
    'data/json/yelp_academic_dataset_tip.json', 
    'data/csv/restaurants_tip.csv', 
    filter_column='business_id'
)

Starting conversion: data/json/yelp_academic_dataset_tip.json -> data/csv/restaurants_tip.csv


Processing yelp_academic_dataset_tip.json: 10it [00:05,  1.98it/s]

Conversion complete! Processed 908915 rows, saved 645894 rows to data/csv/restaurants_tip.csv



In [11]:
# Convert User Data (No filtering needed unless you have a specific list of user_ids)
convert_json_to_csv_chunked(
    'data/json/yelp_academic_dataset_user.json', 
    'data/csv/restaurants_users.csv', 
    filter_column=None
)

Starting conversion: data/json/yelp_academic_dataset_user.json -> data/csv/restaurants_users.csv


Processing yelp_academic_dataset_user.json: 20it [01:03,  3.18s/it]

Conversion complete! Processed 1987897 rows, saved 1987897 rows to data/csv/restaurants_users.csv

